# Lista de Exercícios - Word Embeddings com Gensim

Preencha as células de código com suas soluções para cada exercício.

In [1]:
# Importação da biblioteca necessária
from gensim.models import Word2Vec

/Users/amandafrolini/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Exercício 1 – Explorando o Vocabulário


Treine um modelo Word2Vec com o corpus fornecido:

- Liste o vocabulário
- Mostre o vetor da palavra "rei"
- Informe o tamanho do vetor

In [2]:
sentences = [
    ["rei", "rainha", "homem", "mulher"],
    ["paris", "franca", "roma", "italia"],
    ["homem", "trabalha"],
    ["mulher", "trabalha"],
]


In [3]:
# Treinando o modelo Word2Vec
# min_count=1: Força o modelo a aprender palavras que aparecem apenas 1 vez (essencial para corpus pequeno)
# vector_size=10: Define a dimensão dos vetores (em sala, você pode comparar isso com dimensões maiores como 300)
# window=2: Contexto de palavras vizinhas
model = Word2Vec(sentences, min_count=1, vector_size=10, window=2, seed=42)

# 1. Listar o vocabulário
vocabulario = list(model.wv.key_to_index.keys())
print("Vocabulário do modelo:", vocabulario)

# 2. Mostrar o vetor da palavra "rei"
vetor_rei = model.wv["rei"]
print("\nVetor da palavra 'rei':\n", vetor_rei)

# 3. Informar o tamanho do vetor
tamanho_vetor = len(vetor_rei)
print(f"\nTamanho do vetor: {tamanho_vetor}")

# Observe que cada número dentro do vetor de tamanho 10 representa uma "coordenada" 
# latente da palavra no espaço semântico

Vocabulário do modelo: ['trabalha', 'mulher', 'homem', 'italia', 'roma', 'franca', 'paris', 'rainha', 'rei']

Vetor da palavra 'rei':
 [ 0.08806121 -0.01256962 -0.06786237  0.06653563  0.02595285  0.04005301
 -0.08055468 -0.03752668  0.05356878  0.06645195]

Tamanho do vetor: 10


## Exercício 2 – Similaridade Semântica


- Calcule a similaridade entre:
  - "rei" e "rainha"
  - "rei" e "italia"
- Compare os valores

In [4]:
# Calculando similaridades
sim_rei_rainha = model.wv.similarity("rei", "rainha")
sim_rei_italia = model.wv.similarity("rei", "italia")

print(f"Similaridade entre 'rei' e 'rainha': {sim_rei_rainha:.4f}")
print(f"Similaridade entre 'rei' e 'italia': {sim_rei_italia:.4f}")

Similaridade entre 'rei' e 'rainha': 0.3863
Similaridade entre 'rei' e 'italia': -0.1223


## Exercício 3 – Palavras Mais Próximas


- Encontre as 3 palavras mais similares a "homem"
- Encontre as 3 palavras mais similares a "paris"
- Explique por que os resultados fazem sentido (ou não)

In [5]:
# Encontrando as top 3 palavras mais similares
similares_homem = model.wv.most_similar("homem", topn=3)
similares_paris = model.wv.most_similar("paris", topn=3)

print("Top 3 palavras mais similares a 'homem':")
for palavra, score in similares_homem:
    print(f"- {palavra}: {score:.4f}")

print("\nTop 3 palavras mais similares a 'paris':")
for palavra, score in similares_paris:
    print(f"- {palavra}: {score:.4f}")

Top 3 palavras mais similares a 'homem':
- rei: 0.4424
- italia: 0.4235
- paris: 0.0910

Top 3 palavras mais similares a 'paris':
- italia: 0.2573
- franca: 0.0995
- homem: 0.0910


## Exercício 4 – Analogia Vetorial


Resolva a analogia:

"rei" está para "homem" assim como "rainha" está para ?

Use operações vetoriais.
Interprete o resultado.

In [6]:
# Resolvendo a analogia: "rei" -> "homem" :: "rainha" -> ?
# A operação matemática por trás disso é: vetor(rainha) + vetor(homem) - vetor(rei)
# No GenSim, usamos 'positive' para as adições e 'negative' para as subtrações
resultado_analogia = model.wv.most_similar(positive=["rainha", "homem"], negative=["rei"], topn=1)

print("A palavra que completa a analogia é:")
print(f"- {resultado_analogia[0][0]} (Confiança: {resultado_analogia[0][1]:.4f})")

A palavra que completa a analogia é:
- italia (Confiança: 0.6163)


In [7]:
import numpy as np
import gensim as g

print(f'versão da NumPy: {np.__version__}')
print(f'versão da gensim: {g.__version__}')

versão da NumPy: 1.26.4
versão da gensim: 4.4.0


## Exercício 5 – Atualização do Modelo


- Adicione novas frases ao modelo:
`new_sentences = [
    ["brasil", "capital", "brasilia"],
    ["argentina", "capital", "buenos_aires"]
]`
- Atualize o vocabulário
- Treine novamente
- Verifique se "brasil" está no vocabulário
- Mostre palavras similares a "brasil"

In [8]:
new_sentences = [
    ["brasil", "capital", "brasilia"],
    ["argentina", "capital", "buenos_aires"]
]

# 1. Atualizar o vocabulário com as novas frases
# update=True indica que não queremos descartar o vocabulário anterior
model.build_vocab(new_sentences, update=True)

# 2. Treinar o modelo novamente com as novas frases
# É necessário passar o total de exemplos do modelo para continuar a partir de onde parou
model.train(new_sentences, total_examples=model.corpus_count, epochs=model.epochs)

# 3. Verificar se "brasil" está no vocabulário
esta_no_vocab = "brasil" in model.wv.key_to_index
print(f"'brasil' está no vocabulário? {'Sim' if esta_no_vocab else 'Não'}")
print(model.wv["brasil"])

# 4. Mostrar palavras similares a "brasil"
if esta_no_vocab:
    similares_brasil = model.wv.most_similar("brasil", topn=3)
    print("\nPalavras mais similares a 'brasil':")
    for palavra, score in similares_brasil:
        print(f"- {palavra}: {score:.4f}")

'brasil' está no vocabulário? Sim
[-0.01295667  0.06095286  0.06833837 -0.02250433  0.07961752 -0.04233439
 -0.05208845  0.03649909  0.02735236 -0.07204951]

Palavras mais similares a 'brasil':
- argentina: 0.6425
- trabalha: 0.4119
- brasilia: 0.3931
